<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Grid Search and Pipelines Practice - Solution

---

Work your way through this notebook to practice Grid Search and Pipelines.

## Imports

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

## Load in the dataset from `data/basketball_data.csv`

In [ ]:
bball = pd.read_csv('../data/basketball_data.csv')
bball.head()

Add a target column called `host_wins` that is 1 if the host won the game and 0 if the visiting team won the game (*Hint:* use the `HostName` and `winner` columns).

In [ ]:
bball['host_wins'] = (bball['HostName'] == bball['winner']).astype(int)

## Set up training & testing sets

- The test set will be the 2016 season
- Train is all other seasons

In [ ]:
train = bball[bball['Season'] != 2016]

In [ ]:
test = bball[bball['Season'] == 2016]

## Set up X_train, X_test, y_train, and y_test

`X` will be all columns except the following:
```python
['GameId','GameDate','GameTime','HostName', 'GuestName','total_score','total_line','game_line', 'winner','loser','host_wins','Season']
```

`y` will be whether the home team was the winner (`host_wins`)

In [ ]:
X_train = train.drop(columns = ['GameId','GameDate','GameTime','HostName', 'GuestName','total_score','total_line','game_line', 'winner','loser','host_wins','Season'])
X_test = test.drop(columns = ['GameId','GameDate','GameTime','HostName', 'GuestName','total_score','total_line','game_line', 'winner','loser','host_wins','Season'])
y_train = train['host_wins']
y_test = test['host_wins']

## Calculate baseline accuracy

In [ ]:
y_test.value_counts(normalize = True)

## Create instances of StandardScaler and KNeighborsClassifier to be incorporated into the Pipeline

In [ ]:
ss = StandardScaler()
knn = KNeighborsClassifier()

## Create two pipelines, one for a standalone model, and the other for GridSearchCV

In [ ]:
pipe = Pipeline([
    ('ss', ss),
    ('knn', knn)
])

pipe_gs = Pipeline([
    ('ss', ss),
    ('knn', knn)
])

In [ ]:
pipe_gs_params = {'ss__with_mean': [True, False], 
                 'ss__with_std': [True, False],
                 'knn__p': [1, 2], 
                 'knn__weights': ['uniform', 'distance'],
                 'knn__n_neighbors': [3, 5, 10]}

In [ ]:
pipe.fit(X_train, y_train)

In [ ]:
pipe.score(X_test, y_test)

## `GridSearchCV`

In [ ]:
gs = GridSearchCV(pipe_gs, pipe_gs_params, cv=5, verbose=1)

In [ ]:
gs.fit(X_train, y_train);

In [ ]:
gs.best_score_

In [ ]:
gs.best_params_

In [ ]:
gs.score(X_test, y_test)

## Challenge: 

1. Create a pipeline with `StandardScaler` and `LogisticRegression`
    - For LogisticRegression, set solver to liblinear
2. Run your pipeline through `GridSearchCV`, testing the following parameters:
    - penalty
    - C
    
How does it score on the train and test sets? What are the best parameters?

In [ ]:
pipe_lr = Pipeline([
    ('ss', StandardScaler()),
    ('lr', LogisticRegression(solver = 'liblinear'))
])

In [ ]:
pipe_lr_params = {'lr__penalty': ['l1', 'l2'],
                  'lr__C': [1.0, 0.5, 0.1]}

In [ ]:
gs_lr = GridSearchCV(pipe_lr, pipe_lr_params, cv=5, verbose=1)

In [ ]:
gs_lr.fit(X_train, y_train)

In [ ]:
gs_lr.best_params_

In [ ]:
gs_lr.best_score_

In [ ]:
gs_lr.score(X_test, y_test)